# PA-04 · ¿Qué estado debe devolver cada partición?

**Una misma pregunta, dos formas de ejecutarla** · Reseñas de críticos de cine (Rotten Tomatoes), 2000–2023

Clase 4 · Unidad 2 — ETL y procesamiento escalable

| | |
|---|---|
| **Núcleo de clase** | ≈ 24 min entre trabajo y puesta en común (partes 0 a 6). Reparto orientativo: lectura 2 · inspección 2,5 · declarar 2,5 · ejecutar y comparar 3,5 · decidir e implementar 4,5 · validar 2,5 · defender 1,5 · puesta en común 5 |
| **Para casa** | ≤ 20 min (parte 7) |
| **Qué entregas** | Este mismo notebook, con tus declaraciones, tu expresión y tu explicación completadas |
| **Competencia** | ENTENDER → DECIDIR → IMPLEMENTAR → VALIDAR → DEFENDER |

## El problema

Una plataforma de datos de cine guarda las reseñas de críticos en **24 particiones, una por año (2000–2023)**, en formato Parquet. Cada reseña tiene un veredicto, `fresh` (positiva) o `rotten` (negativa). La pregunta del negocio es:

> **«¿Qué proporción de TODAS las reseñas es `fresh`?»**

Hay dos formas de contestarla:

- **A. Una sola pasada** sobre las 24 particiones.
- **B. Procesar cada partición por separado**: cada una devuelve un resumen pequeño y después se **combinan los 24 resúmenes**.

Un compañero ha hecho la B así: calcula el porcentaje de `fresh` en cada partición y hace la media de los 24 porcentajes.

Tu trabajo **no** es medir rendimiento ni memoria, ni limpiar fechas, ni analizar las reseñas. Es decidir **qué debe devolver cada partición para que la respuesta global se pueda reconstruir correctamente** y **demostrar** que las dos formas de ejecutar contestan a la misma pregunta. Python es un instrumento para obtener evidencia.

## Qué te damos

| Elemento | Qué es |
|---|---|
| `../data/rt_reviews_2000_2023/` | Las reseñas, en Parquet particionado por año (una carpeta `year=AAAA` por partición) |
| `../data/DATASET_NOTICE.md` | Procedencia, qué se ha seleccionado y qué no se ha tocado |
| Código del compañero (parte 1) | El que calcula la media de los 24 porcentajes |
| Funciones ya escritas | La lectura perezosa, el cálculo de resúmenes por partición y las comprobaciones |

> **Código proporcionado — no se evalúa esta sintaxis.** Varias celdas usan `scan_parquet`, `group_by`, `agg` o expresiones de Polars. No tienes que escribirlas ni memorizarlas: **escribes valores, listas, frases cortas y una única expresión sencilla** (parte 4). Sí eres responsable de **entender qué consigue el código, qué devuelve cada partición y qué significa el resultado**.

### GLOSARIO ELABORADO PARA ESTA ACTIVIDAD

Los ficheros **no traen documentación**; este glosario **no es documentación original del conjunto de datos**: lo hemos elaborado solo con lo que necesitas para leer la actividad.

| Nombre | Qué es |
|---|---|
| `reviewState` | Veredicto de la reseña: `fresh` o `rotten`. En esta actividad, *positiva* = `fresh` |
| `year` | Año de la partición (la carpeta `year=AAAA`) |

## Cómo se trabaja: DECLARA, LUEGO VERIFICA

1. **Declaras** lo que esperas *antes* de ejecutar la comprobación.
2. **Verificas** con una comprobación ejecutable.
3. Si el resultado te sorprende, esa sorpresa es lo valioso: apunta por qué.

## Sobre el uso de IA

Puedes usar documentación externa y herramientas de IA. **La IA puede generar; tú sigues siendo responsable** de entender, comprobar, poner a prueba y defender lo que entregas. Un código que se ejecuta y da una cifra verosímil no está por eso validado.

---
## Parte 0 · Preparación

Ejecuta la celda. No hay nada que decidir aquí.

In [ ]:
# SOLO PARA GOOGLE COLAB (añadido para el repositorio; en Jupyter o VS Code esta celda no hace nada).
# Descarga desde GitHub la carpeta data/ de esta práctica y se sitúa en student/,
# para que las rutas "../data/..." del notebook funcionen igual que en local.
import os, sys
if "google.colab" in sys.modules:
    import urllib.parse, urllib.request
    BASE = "https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/BigData/PA-04/"
    FICHEROS = [
        'data/rt_reviews_2000_2023/year=2000/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2001/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2002/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2003/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2004/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2005/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2006/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2007/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2008/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2009/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2010/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2011/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2012/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2013/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2014/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2015/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2016/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2017/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2018/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2019/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2020/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2021/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2022/00000000.parquet',
        'data/rt_reviews_2000_2023/year=2023/00000000.parquet',
    ]
    for f in FICHEROS:
        destino = os.path.join("/content/PA-04", f)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        if not os.path.exists(destino):
            urllib.request.urlretrieve(BASE + urllib.parse.quote(f), destino)
    os.makedirs("/content/PA-04/student", exist_ok=True)
    os.chdir("/content/PA-04/student")
    print(f"Colab: {len(FICHEROS)} ficheros de datos listos en /content/PA-04/data")

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve cada partición y qué significa el resultado)

import math

import polars as pl

RUTA = "../data/rt_reviews_2000_2023"


def comprobar(condicion, texto):
    """Como un assert, pero no detiene el notebook: imprime ✔ si se cumple y ✘ si no."""
    print(("✔ " if condicion else "✘ ") + texto)


print("polars", pl.__version__)
if pl.__version__ != "1.44.2":
    print("Aviso: esta actividad se ha verificado con polars 1.44.2; con otras versiones algunos resultados pueden diferir.")

lf = pl.scan_parquet(RUTA + "/**/*.parquet", hive_partitioning=True)       # lectura perezosa: no carga nada hasta .collect()

---
## Parte 1 · ENTENDER: las particiones y el código del compañero (≈ 4,5 min)

### 1.1 Qué hay en el almacén

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve cada partición y qué significa el resultado)

print("Columnas:", dict(lf.collect_schema()))
tamanos = lf.group_by("year").agg(pl.len().alias("n_resenas")).sort("year").collect()
print("Particiones:", tamanos.height)
with pl.Config(tbl_rows=30):
    print(tamanos)

### 1.2 El código del compañero

Esto es lo que ejecutó el compañero. **Todavía no se ha ejecutado aquí.**

```python
por_particion = lf.group_by("year").agg((pl.col("reviewState") == "fresh").mean().alias("porcentaje_fresh"))      # [1] % de fresh en cada partición
cifra_del_colega = por_particion.select(pl.col("porcentaje_fresh").mean()).collect().item()                          # [2] media de los 24 porcentajes
```

---
## Parte 2 · DECIDIR (declarar): antes de ejecutar nada (≈ 2,5 min)

Escribe tus respuestas **antes** de ver ningún resultado.

**2.1 Predicción.** ¿Crees que la cifra del compañero coincidirá con la proporción de `fresh` sobre **todas** las reseñas? Responde `"sí"`, `"no"` o `"no sé"` y explica por qué en una frase.

In [ ]:
prediccion = {
    "coinciden": "",
    "porque": "",
}

**2.2 El estado.** En la forma B cada partición se procesa por separado y devuelve **un resumen**. Elige **qué nombres** debe devolver cada partición para que, con los 24 resúmenes, se pueda **reconstruir la proporción global de `fresh`** exactamente. Escribe la lista de nombres entre estos:

| Nombre | Qué calcula cada partición |
|---|---|
| `total_count` | Número de reseñas de la partición |
| `positive_count` | Número de reseñas `fresh` de la partición |
| `negative_count` | Número de reseñas `rotten` de la partición |
| `positive_share` | Proporción de `fresh` de la partición (`fresh` ÷ reseñas) |

In [ ]:
estado_que_debe_devolver_cada_particion = []

**2.3 Cómo comparar.** Cuando tengas las dos respuestas (A y B), ¿cómo las vas a comparar? Para los **conteos enteros** y para las **proporciones decimales**, elige `"igualdad exacta"` o `"con tolerancia"`.

In [ ]:
como_comparo = {
    "conteos_enteros": "",
    "proporciones_decimales": "",
}

---
## Parte 3 · IMPLEMENTAR y VERIFICAR: dos formas de ejecutar (≈ 3,5 min)

### 3.1 Las dos respuestas

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve cada partición y qué significa el resultado)

# --- Forma B del compañero ---
por_particion = lf.group_by("year").agg((pl.col("reviewState") == "fresh").mean().alias("porcentaje_fresh"))
cifra_del_colega = por_particion.select(pl.col("porcentaje_fresh").mean()).collect().item()

# --- Forma A: una sola pasada (referencia) ---
referencia = lf.select((pl.col("reviewState") == "fresh").sum().alias("positive_count"), pl.len().alias("total_count")).collect()
positive_ref, total_ref = int(referencia["positive_count"][0]), int(referencia["total_count"][0])
proporcion_ref = positive_ref / total_ref

print("Los dos códigos se han ejecutado sin errores ni avisos.")
print(f"Forma B (media de los {tamanos.height} porcentajes): {100 * cifra_del_colega:.4f} %")
print(f"Forma A (una sola pasada, {total_ref} reseñas): {100 * proporcion_ref:.4f} %")
print(f"Diferencia B − A: {100 * (cifra_del_colega - proporcion_ref):+.3f} puntos porcentuales")
if prediccion["coinciden"]:
    coinciden = math.isclose(cifra_del_colega, proporcion_ref, rel_tol=1e-9)
    comprobar(prediccion["coinciden"] == ("sí" if coinciden else "no"), f"Declaraste «{prediccion['coinciden']}»; las dos respuestas {'coinciden' if coinciden else 'no coinciden'}")
else:
    print("✘ (pendiente) Declara primero tu predicción en 2.1 para contrastarla.")

### 3.2 Qué pesa cada partición

Para cada partición: cuántas reseñas tiene, su proporción de `fresh`, qué parte del total de reseñas representa y qué peso tiene en la media de los 24 porcentajes.

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve cada partición y qué significa el resultado)

evidencia = lf.group_by("year").agg(pl.len().alias("n_resenas"), (pl.col("reviewState") == "fresh").sum().alias("n_fresh")).sort("year").collect()
evidencia = evidencia.with_columns(
    (100 * pl.col("n_fresh") / pl.col("n_resenas")).round(2).alias("% fresh"),
    (100 * pl.col("n_resenas") / pl.col("n_resenas").sum()).round(2).alias("% del total de reseñas"),
    (pl.lit(100.0) / evidencia.height).round(2).alias("peso en la media simple (%)"),
)
with pl.Config(tbl_rows=30, tbl_cols=6, tbl_width_chars=150):
    print(evidencia.select("year", "n_resenas", "% fresh", "% del total de reseñas", "peso en la media simple (%)"))

> **Pregunta silenciosa.** El código del compañero no da error y su cifra es verosímil. ¿Qué ha comprobado realmente? Mira la partición más pequeña y la más grande: ¿con qué peso cuenta cada una en la cifra del compañero y con cuál en la pregunta «qué proporción de **todas** las reseñas es `fresh`»?

---
## Parte 4 · DECIDIR e IMPLEMENTAR: el estado y su combinación (≈ 4,5 min)

### 4.1 Los resúmenes de cada partición

Se te da una función que, para **los nombres que tú has declarado**, devuelve **un resumen por partición** (una fila por año). Ejecútala con tu declaración de 2.2:

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve cada partición y qué significa el resultado)

AGREGADOS = {
    "total_count": pl.len(),
    "positive_count": (pl.col("reviewState") == "fresh").sum(),
    "negative_count": (pl.col("reviewState") == "rotten").sum(),
    "positive_share": (pl.col("reviewState") == "fresh").mean(),
}


def calcular_estado(lazy, nombres):
    """Una fila por partición (year) con los resúmenes que se piden, calculados partición a partición."""
    return lazy.group_by("year").agg([AGREGADOS[n].alias(n) for n in nombres]).sort("year").collect()


estado = None
if not estado_que_debe_devolver_cada_particion:
    print("✘ (pendiente) Declara primero la lista de 2.2.")
elif not all(n in AGREGADOS for n in estado_que_debe_devolver_cada_particion):
    print("✘ Nombres no válidos:", [n for n in estado_que_debe_devolver_cada_particion if n not in AGREGADOS], "· válidos:", list(AGREGADOS))
else:
    estado = calcular_estado(lf, estado_que_debe_devolver_cada_particion)
    print(f"Estado devuelto: {estado.height} filas (una por partición) y las columnas {estado.columns}")
    print(estado.head(5))

### 4.2 Tu combinación

`estado` es una tabla de Polars con **una fila por partición**. Con una columna, por ejemplo `estado["columna"].sum()`, se suman sus valores; con `/` se divide. Escribe **una sola expresión** que **combine los 24 resúmenes** y dé **la proporción global de `fresh`** (un número entre 0 y 1). Usa solo las columnas que has pedido en 2.2. *(Si da error, lee el mensaje: suele ser un nombre de columna que no pediste.)*

In [ ]:
proporcion_reconstruida = None

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve cada partición y qué significa el resultado)

if proporcion_reconstruida is None:
    print("✘ (pendiente) Escribe tu expresión en la celda anterior.")
else:
    print(f"Proporción reconstruida con tus {estado.height} resúmenes: {100 * proporcion_reconstruida:.4f} %")

---
## Parte 5 · VALIDAR: ¿las dos formas contestan a la misma pregunta? (≈ 2,5 min)

Comparación con el método que declaraste en 2.3.

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve cada partición y qué significa el resultado)

def comparar(valor, esperado, metodo):
    """True/False según el método declarado: igualdad exacta o tolerancia relativa 1e-9."""
    if metodo == "igualdad exacta":
        return valor == esperado
    if metodo == "con tolerancia":
        return math.isclose(valor, esperado, rel_tol=1e-9, abs_tol=0.0)
    return None


if proporcion_reconstruida is None or not all(como_comparo.values()):
    print("✘ (pendiente) Necesitas tu expresión (4.2) y tu método de comparación (2.3).")
else:
    referencia_conteos = {"positive_count": positive_ref, "total_count": total_ref}
    suma_de_tus_resumenes = {n: int(estado[n].sum()) for n in estado.columns if n in referencia_conteos}
    for n, v in suma_de_tus_resumenes.items():
        comprobar(comparar(v, referencia_conteos[n], como_comparo["conteos_enteros"]), f"Conteo «{n}» ({como_comparo['conteos_enteros']}): suma de tus resúmenes {v} frente a la forma A {referencia_conteos[n]}")
    if not suma_de_tus_resumenes:
        print("— No has pedido ningún conteo de la forma A (positive_count, total_count): no hay conteos enteros que comparar.")
    ok = comparar(float(proporcion_reconstruida), proporcion_ref, como_comparo["proporciones_decimales"])
    comprobar(ok, f"Proporción ({como_comparo['proporciones_decimales']}): tu reconstrucción {100 * float(proporcion_reconstruida):.6f} % frente a la forma A {100 * proporcion_ref:.6f} % (diferencia {abs(float(proporcion_reconstruida) - proporcion_ref):.2e})")
    print()
    print(f"Para comparar, la cifra del compañero (forma B): {100 * cifra_del_colega:.6f} % frente a {100 * proporcion_ref:.6f} % (diferencia {abs(cifra_del_colega - proporcion_ref):.2e}; {'coinciden' if math.isclose(cifra_del_colega, proporcion_ref, rel_tol=1e-9) else 'no coinciden con ninguna tolerancia razonable'})")

> **Pregunta silenciosa.** ¿Habrías detectado el error del compañero sin la forma A? ¿Qué te dice que dos formas de ejecutar contestan a la misma pregunta: que las dos cifras sean parecidas, o que se pueda reconstruir una desde los resúmenes de la otra?

---
## Parte 6 · DEFENDER: tu explicación (≈ 1,5 min)

**6.1 Explica en dos frases cortas** (para alguien que no ha visto el notebook):

- `que_debe_devolver_cada_particion`: qué debe devolver cada partición y por qué eso basta para reconstruir la respuesta global.
- `por_que_no_basta_el_porcentaje`: por qué la media de los 24 porcentajes no es la proporción global.

**6.2 Tus dos cifras**, en %, con dos decimales, **de tu propia ejecución**: la de la forma B del compañero y la global.

In [ ]:
mi_explicacion = {
    "que_debe_devolver_cada_particion": "",
    "por_que_no_basta_el_porcentaje": "",
}
mis_cifras = {
    "cifra_del_colega_pct": None,
    "cifra_global_pct": None,
}

In [ ]:
if mis_cifras["cifra_del_colega_pct"] is None or mis_cifras["cifra_global_pct"] is None:
    print("✘ (pendiente) Escribe tus dos cifras en 6.2.")
else:
    comprobar(abs(mis_cifras["cifra_del_colega_pct"] - 100 * cifra_del_colega) < 0.006, "«cifra_del_colega_pct» coincide con tu ejecución")
    comprobar(abs(mis_cifras["cifra_global_pct"] - 100 * proporcion_ref) < 0.006, "«cifra_global_pct» coincide con tu ejecución")
print()
print("Estado de tu entrega:")
print(" · predicción declarada:", all(bool(v) for v in prediccion.values()))
print(" · estado declarado:", bool(estado_que_debe_devolver_cada_particion))
print(" · método de comparación declarado:", all(bool(v) for v in como_comparo.values()))
print(" · combinación escrita:", proporcion_reconstruida is not None)
print(" · explicación escrita:", all(bool(v) for v in mi_explicacion.values()))
print(" · cifras escritas:", all(v is not None for v in mis_cifras.values()))

---
# ⏸ PARADA DEL NÚCLEO DE CLASE

**Deja de escribir.** Lo que queda (parte 7) es para casa. Ahora toca la **puesta en común** (≈ 5 min). Ten a mano:

1. Tu predicción de 2.1 y lo que ha ocurrido.
2. Qué devuelve cada una de tus particiones y con qué lo has combinado.
3. Qué te permite afirmar que las dos formas de ejecutar contestan a la misma pregunta.
4. Por qué la cifra del compañero, aunque verosímil, no responde a la pregunta global.

---
## Parte 7 · Para casa (≤ 20 min): comprueba y defiende

**Qué no se hace aquí:** nada nuevo en cuanto a conceptos. Es comprobar tu estado con otra agrupación y dejar por escrito lo que ya has razonado en clase.

### 7.1 ¿Funciona con otra agrupación de particiones?
Agrupa las particiones en **dos mitades** (2000–2011 y 2012–2023). Para cada mitad se calcula su resumen (la mitad hace ahora de «partición») y se combinan las dos mitades de dos maneras: **sumando los conteos** y **promediando los dos porcentajes**. **Antes de ejecutar**, declara para cada manera si esperas que coincida con la forma A (`"coincide"` o `"no coincide"`).

In [ ]:
prediccion_agrupacion = {
    "sumando_los_conteos": "",
    "promediando_los_porcentajes": "",
}

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve cada partición y qué significa el resultado)

def estado_de(lazy):
    """Resumen de un conjunto de particiones tratado como si fuese una sola partición."""
    return lazy.select(AGREGADOS["positive_count"].alias("positive_count"), AGREGADOS["total_count"].alias("total_count")).collect().row(0, named=True)


mitades = {"2000–2011": lf.filter(pl.col("year") <= 2011), "2012–2023": lf.filter(pl.col("year") >= 2012)}
por_mitad = {nombre: estado_de(mitad) for nombre, mitad in mitades.items()}
for nombre, fila in por_mitad.items():
    print(nombre, fila, f"→ {100 * fila['positive_count'] / fila['total_count']:.4f} %")
suma = sum(f["positive_count"] for f in por_mitad.values()) / sum(f["total_count"] for f in por_mitad.values())
promedio = sum(f["positive_count"] / f["total_count"] for f in por_mitad.values()) / len(por_mitad)
print(f"Sumando los conteos: {100 * suma:.4f} %   · promediando los dos porcentajes: {100 * promedio:.4f} %   · forma A: {100 * proporcion_ref:.4f} %")
if all(prediccion_agrupacion.values()):
    comprobar(prediccion_agrupacion["sumando_los_conteos"] == ("coincide" if suma == proporcion_ref else "no coincide"), f"Sumando los conteos: declaraste «{prediccion_agrupacion['sumando_los_conteos']}»")
    comprobar(prediccion_agrupacion["promediando_los_porcentajes"] == ("coincide" if math.isclose(promedio, proporcion_ref, rel_tol=1e-9) else "no coincide"), f"Promediando los porcentajes: declaraste «{prediccion_agrupacion['promediando_los_porcentajes']}»")
else:
    print("✘ (pendiente) Declara antes tu predicción en la celda anterior.")

In [ ]:
resumen_agrupacion = ""     # 2-3 frases: ¿qué esperabas?, ¿qué ha pasado con cada manera?, ¿qué te dice sobre «combinar resúmenes»?

### 7.2 Tu párrafo final
Escribe en un párrafo (4-6 líneas): **qué debe devolver cada partición, por qué la media de los porcentajes no responde a la pregunta global y cómo has comprobado que las dos formas de ejecutar son equivalentes.**

In [ ]:
parrafo_final = ""

### 7.3 (opcional, no se entrega) ¿Siempre se nota el error?
Elige **una lista de años** (al menos dos) y mira la cifra del compañero y la global **solo con esos años**. ¿Puedes encontrar una lista donde casi coincidan? Si coinciden, ¿es correcto el método del compañero?

In [ ]:
mis_anos = []

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve cada partición y qué significa el resultado)

if len(mis_anos) < 2:
    print("✘ (pendiente) Escribe al menos dos años en la celda anterior si quieres probarlo (no es obligatorio).")
else:
    todos = calcular_estado(lf, ["positive_count", "total_count"]).filter(pl.col("year").is_in(mis_anos))
    if todos.height < 2:
        print("✘ Alguno de esos años no está entre las 24 particiones (2000–2023).")
    else:
        colega = (todos["positive_count"] / todos["total_count"]).mean()
        global_ = todos["positive_count"].sum() / todos["total_count"].sum()
        print(f"{todos.height} particiones; tamaños de {todos['total_count'].min()} a {todos['total_count'].max()} reseñas")
        print(f"Cifra del compañero: {100 * colega:.4f} %  · global: {100 * global_:.4f} %  · diferencia {100 * (colega - global_):+.3f} pp")

In [ ]:
mi_observacion_opcional = ""

In [ ]:
print("Estado de tu entrega (casa):")
print(" · predicción de la agrupación:", all(bool(v) for v in prediccion_agrupacion.values()))
print(" · resumen de la agrupación:", bool(resumen_agrupacion))
print(" · párrafo final:", bool(parrafo_final))

### Lista de verificación antes de entregar
- [ ] Declaré mi predicción, el estado y el método de comparación **antes** de ejecutar.
- [ ] Sé decir qué devuelve cada partición y por qué eso basta.
- [ ] Sé explicar con qué peso cuenta cada partición en cada cifra.
- [ ] Mi comprobación de equivalencia no se apoya solo en que las cifras se parezcan.
- [ ] Mis cifras salen de mi propia ejecución.
- [ ] Si he usado IA o documentación externa, entiendo lo que he copiado y lo he comprobado sobre estos datos.

### Para quien quiera más (opcional)
- Escribe, con tus palabras, una regla de una frase para decidir si un resumen de una partición se puede combinar con los de las demás.
- Piensa otra magnitud (un máximo, una mediana, un recuento de valores distintos) y decide qué debería devolver cada partición para combinarla.